# 75 Unity Catalog · Jerarquía y Permisos

Vas a construir la tabla gobernada `padron`, recorrer el namespace de Unity Catalog y practicar el ciclo de vida y los privilegios de sus objetos. Los objetivos DCEA son **diferenciar managed y external tables**, **aplicar GRANT/REVOKE** e **identificar roles clave de Unity Catalog**.

In [ ]:
CATALOG = "big_data_ii_2025"
SCHEMA  = "spark_examples"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Trabajando en {CATALOG}.{SCHEMA}")

In [ ]:
display(spark.sql("SELECT current_user() AS usuario, current_catalog() AS catalogo, current_schema() AS esquema"))

## Namespace de Unity Catalog

```text
Metastore
└── Catalog
    └── Schema
        ├── Table
        ├── View
        ├── Volume
        ├── Function
        └── Model
```

Recorré los resultados de general a específico y fijate que el nombre completo de un objeto sigue la forma `catalog.schema.object`.

In [ ]:
display(spark.sql("SHOW CATALOGS"))
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA}"))

## Construcción del padrón electoral

Los archivos del TSE no tienen encabezado, usan `ISO-8859-1` y contienen campos de ancho fijo. Por eso vas a declarar todos los campos como `STRING` y aplicar `trim()` antes de cruzarlos. Aunque el padrón es público, `cedula` y `nombre_completo` siguen siendo PII.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

RUTA_PADRON = f"/Volumes/{CATALOG}/{SCHEMA}/spark_data/PADRON_COMPLETO.csv"
RUTA_DISTELEC = f"/Volumes/{CATALOG}/{SCHEMA}/spark_data/distelec.csv"

def exigir_archivo(ruta):
    try:
        dbutils.fs.ls(ruta)
    except Exception as e:
        raise FileNotFoundError(
            f"No se encontró el archivo requerido: {ruta}.\n"
            f"Cargá los datos del TSE en el volumen antes de continuar. Detalle: {e}")

exigir_archivo(RUTA_PADRON)
exigir_archivo(RUTA_DISTELEC)

esquema_padron = StructType([StructField(nombre, StringType(), True) for nombre in [
    "cedula", "codelec", "relleno", "fecha_caducidad_raw",
    "junta_raw", "nombre", "apellido1", "apellido2"]])
esquema_distelec = StructType([StructField(nombre, StringType(), True) for nombre in [
    "codelec", "provincia", "canton", "distrito"]])

padron_raw = (spark.read.option("header", False).option("encoding", "ISO-8859-1")
    .schema(esquema_padron).csv(RUTA_PADRON))
distelec_raw = (spark.read.option("header", False).option("encoding", "ISO-8859-1")
    .schema(esquema_distelec).csv(RUTA_DISTELEC))

def trim_textos(df):
    return df.select(*[F.trim(F.col(c)).alias(c) for c in df.columns])

padron_limpio = trim_textos(padron_raw)
distelec_limpio = trim_textos(distelec_raw).select(
    "codelec",
    F.initcap("provincia").alias("provincia"),
    F.initcap("canton").alias("canton"),
    F.initcap("distrito").alias("distrito"))

In [ ]:
padron = (padron_limpio.alias("p")
    .join(distelec_limpio.alias("d"), on="codelec", how="left")
    .select(
        F.col("cedula"),
        F.col("codelec"),
        F.to_date("fecha_caducidad_raw", "yyyyMMdd").alias("fecha_caducidad"),
        F.col("junta_raw").cast("int").alias("junta"),
        F.col("nombre"), F.col("apellido1"), F.col("apellido2"),
        F.concat_ws(" ", "nombre", "apellido1", "apellido2").alias("nombre_completo"),
        F.substring("codelec", 1, 1).alias("provincia_codigo"),
        F.substring("codelec", 2, 2).alias("canton_codigo"),
        F.substring("codelec", 4, 3).alias("distrito_codigo"),
        F.col("d.provincia").alias("provincia"),
        F.col("d.canton").alias("canton"),
        F.col("d.distrito").alias("distrito")))

validacion = padron.agg(
    F.count(F.lit(1)).alias("filas"),
    F.sum(F.when(F.col("provincia").isNull(), 1).otherwise(0)).alias("sin_provincia"),
    F.countDistinct("cedula").alias("cedulas_distintas")).first()
filas = int(validacion["filas"])
sin_provincia = int(validacion["sin_provincia"] or 0)
duplicados_cedula = filas - int(validacion["cedulas_distintas"])

assert filas > 3_000_000, f"El padrón solo contiene {filas:,} filas; se esperaban más de 3.000.000."
assert sin_provincia == 0, f"Quedaron {sin_provincia:,} filas sin provincia después del join."
assert duplicados_cedula == 0, f"Se detectaron {duplicados_cedula:,} cédulas duplicadas."
print(f"Validación superada: {filas:,} filas, 0 sin provincia y 0 cédulas duplicadas.")

In [ ]:
TABLA_PADRON = f"{CATALOG}.{SCHEMA}.padron"
(padron.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(TABLA_PADRON))
print(f"Tabla managed creada: {TABLA_PADRON}")
display(spark.table(TABLA_PADRON).limit(10))

## Managed vs external

En `DESCRIBE EXTENDED`, revisá `Type`, `Location` y `Provider`. Una managed table delega a Unity Catalog tanto los metadatos como el ciclo de vida de los archivos; una external table gobierna metadatos, pero conserva los archivos en una ubicación externa. Free Edition no permite crear external locations propias, así que estudiamos la diferencia mediante los metadatos disponibles.

In [ ]:
display(spark.sql(f"DESCRIBE EXTENDED {TABLA_PADRON}"))
display(spark.sql(f"""
SELECT table_name, table_type, data_source_format
FROM {CATALOG}.information_schema.tables
WHERE table_schema = '{SCHEMA}'
ORDER BY table_name
"""))

## DROP y UNDROP

Unity Catalog conserva temporalmente una managed table eliminada para permitir `UNDROP`. Usá una tabla desechable y fijate cómo reaparece sin reconstruir sus datos.

In [ ]:
TABLA_PRUEBA = f"{CATALOG}.{SCHEMA}.tabla_prueba_drop"
spark.sql(f"CREATE OR REPLACE TABLE {TABLA_PRUEBA} USING DELTA AS SELECT 1 AS id")
spark.sql(f"DROP TABLE {TABLA_PRUEBA}")
display(spark.sql(f"SHOW TABLES DROPPED IN {CATALOG}.{SCHEMA}"))
spark.sql(f"UNDROP TABLE {TABLA_PRUEBA}")
assert spark.catalog.tableExists(TABLA_PRUEBA), "UNDROP no restauró la tabla de prueba."
print("DROP/UNDROP verificado.")

## Privilegios sobre objetos

Primero inspeccioná los grants efectivos. Si ya construiste el Lab A, la demostración concede y revoca `SELECT` sobre `movielens_gold_genero`; si todavía no existe, el notebook lo indica y conserva el padrón sin ampliar su exposición.

In [ ]:
display(spark.sql(f"SHOW GRANTS ON TABLE {TABLA_PADRON}"))
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOG}.{SCHEMA}"))

TABLA_GOLD = f"{CATALOG}.{SCHEMA}.movielens_gold_genero"
if spark.catalog.tableExists(TABLA_GOLD):
    spark.sql(f"GRANT SELECT ON TABLE {TABLA_GOLD} TO `account users`")
    display(spark.sql(f"SHOW GRANTS ON TABLE {TABLA_GOLD}"))
    spark.sql(f"REVOKE SELECT ON TABLE {TABLA_GOLD} FROM `account users`")
    print("GRANT y REVOKE aplicados sobre movielens_gold_genero.")
else:
    print(
        f"No existe {TABLA_GOLD}; ejecutá el Lab A para repetir la demostración de GRANT/REVOKE "
        "sobre ese objeto. No se otorgó acceso grupal al padrón con PII.")

display(spark.sql(f"""
SELECT grantor, grantee, privilege_type, table_name
FROM {CATALOG}.information_schema.table_privileges
WHERE table_schema = '{SCHEMA}'
ORDER BY table_name, grantee, privilege_type
"""))

## Nota de examen

En Unity Catalog **no existe `DENY`**: el modelo es *deny-by-default* y se administra con `GRANT` y `REVOKE`. `DENY` es una sentencia heredada que aplica únicamente a `hive_metastore`. Aunque la guía del examen menciona los tres verbos, en Unity Catalog retirás acceso mediante `REVOKE`.

Los roles clave son el **metastore admin**, el **owner** de cada objeto y quien recibe el privilegio **MANAGE**. `MANAGE` permite administrar permisos sin transferir la propiedad del objeto.

## Cierre

- Recorriste la jerarquía Metastore → Catalog → Schema → Object.
- Construiste y validaste el padrón real como managed Delta table.
- Comparaste managed y external tables mediante metadatos.
- Verificaste `DROP`/`UNDROP` y consultaste privilegios.
- Relacionaste `GRANT`, `REVOKE`, ownership y `MANAGE` con el modelo deny-by-default.